# Day 18: Loading and Cleaning Data with pandas
Objectives:
- Read and write CSV with explicit dtypes, missing-value markers, and dates.
- Handle missing values and type conversions.
- Parse dates and set index.
- Robust cleaning patterns (pipe).


**Offline teaching data:** These examples use original synthetic fixtures from the [reviewed fixture collection](../../fixtures/README.md). They are not the remote Seaborn sample tables or historical observations. The loader makes no download; inference and model scores describe the teaching fixture only.


<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-18`. Read
`python/ds-60day/companion-guides/day18_pandas_io_cleaning.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

Reading a file creates a DataFrame but does not prove its schema.
External columns may have missing values, inconsistent spellings,
numeric text, impossible values, or duplicate records. Profile those
conditions before converting types, because conversion can hide which
representations arrived.

A cleaning function should accept a raw frame, work on a copy, apply
named deterministic steps, and return a clean frame. Record why rows or
values change. Prefer nullable pandas dtypes where absence is valid, and
make the function idempotent when practical: cleaning already-clean
data should not keep changing it.

### Vocabulary

- **missing value:** an absent observation represented by pandas missing markers.
- **dtype:** a column's stored representation and operation rules.
- **coercion:** conversion that may replace unparseable values with missing data.
- **duplicate:** a repeated row or repeated key under a stated definition.
- **idempotent:** producing the same result when applied again to its own output.
- **data lineage:** evidence about where data came from and how it changed.

## Syntax anatomy

`pd.to_numeric(series, errors="coerce")` converts compatible text and
marks failures missing; those new missing values must be counted and
reviewed. `.assign(...)` returns a new frame with derived/replaced
columns. `.pipe(clean_step)` passes a DataFrame into a named function,
making a sequence of transformations readable.

### Worked example 1: Normalize text and numeric representations

Count conversion failures instead of silently losing them. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "name": [" Ada ", "Lin", "Grace"],
        "score": ["10", "missing", "8.5"],
    }
)
clean_scores = pd.to_numeric(raw["score"], errors="coerce")
cleaned = raw.assign(
    name=raw["name"].str.strip(),
    score=clean_scores,
)
(cleaned.to_dict("records"), int(cleaned["score"].isna().sum()))

**Expected observation:** The names are trimmed, numeric text is converted, and one conversion failure is reported as missing.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2: Write a cleaning function that preserves raw input

Copy at the boundary and make repeated application stable. Predict first; then run the next cell.

In [ ]:
def clean_people(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["name"] = result["name"].str.strip()
    result["score"] = pd.to_numeric(result["score"], errors="coerce")
    return result.drop_duplicates().reset_index(drop=True)


once = clean_people(raw)
twice = clean_people(once)
(raw.loc[0, "name"], once.equals(twice))

**Expected observation:** `(' Ada ', True)`. The raw frame is unchanged and the demonstrated cleaner is idempotent on this data.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Profile raw dtypes, missing counts, unique spellings, and duplicate keys before cleaning.
2. After coercion, count values that became missing and retain examples for review.
3. Avoid `inplace=True` while learning; returned frames make ownership and chaining clearer.
4. Re-read saved output and reconcile row count, schema, and key totals with the in-memory clean frame.

**Alternative to compare:** Use pandas for tabular batch cleaning, the `csv` module for simple streaming, and a database when constraints/transactions belong at storage.

**Boundary to test:** Blank strings versus nulls, locale-formatted numbers, duplicate keys with conflicting fields, empty input, and dtype drift need policy.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

## Read and write CSV with explicit parsing rules

`pd.read_csv` guesses types unless you state them. The example reads CSV text
from memory (`StringIO` behaves like an open file), names the text columns'
dtype, declares that `n/a` means missing, and parses the date column. It then
writes the frame back to CSV and reads it again with the same rules to prove
the round trip keeps every value and type.

Predict the three values in the final tuple before running the next cell.

In [ ]:
from io import StringIO

import pandas as pd

csv_text = """order_id,amount,city,ordered_on
A1,10.5,SF,2025-01-03
A2,n/a,NY,2025-01-04
A3,7,ny,2025-01-05
"""
rules = {"dtype": {"order_id": "string", "city": "string"}, "parse_dates": ["ordered_on"]}
orders = pd.read_csv(StringIO(csv_text), na_values=["n/a"], **rules)

buffer = StringIO()
orders.to_csv(buffer, index=False)
reloaded = pd.read_csv(StringIO(buffer.getvalue()), **rules)
(orders.dtypes.astype(str).to_dict(), int(orders["amount"].isna().sum()), reloaded.equals(orders))

**Expected observation:**
`({'order_id': 'string', 'amount': 'float64', 'city': 'string', 'ordered_on': 'datetime64[ns]'}, 1, True)`.
Exactly one amount became missing because `n/a` was declared as a missing
marker, and the written CSV stores it as an empty field, which the reader
treats as missing again. Without `dtype=`, the text columns would be `object`;
without `parse_dates=`, `ordered_on` would stay text. `city` still contains
both `NY` and `ny`: reading a file does not clean it, which is the next step.

The next cell loads the course's `passenger-outcomes` fixture, which is already parsed, and profiles it with `df.info()` before showing the first rows.

In [ ]:
from ds60sqlpy.teaching_data import load_teaching_frame

df = load_teaching_frame("passenger-outcomes")
df.info()
df.head()

## Handling missing and types
- isna/fillna/dropna
- astype and to_numeric
- parse_dates in read_* functions


In [ ]:
missing_age_share = df["age"].isna().mean()
filled_preview = df["age"].fillna(df["age"].median()).head()
# A missing age stays missing (<NA>) instead of silently becoming "not adult".
df["adult"] = df["age"].ge(18).astype("boolean").where(df["age"].notna())
df["fare"] = pd.to_numeric(df["fare"], errors="coerce")
print("share of missing ages:", round(float(missing_age_share), 3))
print(filled_preview)
df[["age", "fare", "adult"]].dtypes

## Build a cleaning function and use .pipe()
Encapsulate logic for reuse and testing.

In [ ]:
def clean_passengers(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["age_missing"] = result["age"].isna()
    result["adult"] = result["age"].ge(18).astype("boolean").where(result["age"].notna())
    result["age"] = result["age"].fillna(result["age"].median())
    result["fare"] = pd.to_numeric(result["fare"], errors="coerce")
    result["fare"] = result["fare"].fillna(result["fare"].median())
    return result


tidy = df.pipe(clean_passengers)
tidy[["age", "age_missing", "adult", "fare"]].head()

## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

**Dataset:** the exercises use the offline `passenger-outcomes` fixture,
loaded with `from ds60sqlpy.teaching_data import load_teaching_frame` and
`df = load_teaching_frame("passenger-outcomes")`. Its 360 synthetic rows have
15 columns, including `survived`, `pclass`, `sex`, `age`, `fare`, `embarked`, and
`deck`; `age` is missing in 28 rows, `fare` in 12, `embarked` in 10, and `deck`
in 270.

1. Load a local CSV into pandas, recording path, encoding, row grain, shape, columns, dtypes, missing counts, and duplicate-key counts before changing anything.
   **Expected behavior:** produce a compact profile, not a full data dump. **Constraint:** use repository-relative `Path` objects and no network source.
   **Verify:** save the first profile values, restart the kernel, and assert the second shape, columns, dtypes, missing counts, and duplicate-key counts exactly match.

2. Implement `clean_frame(raw)` that trims selected text, converts documented numeric/date fields, handles missing values by written policy, resolves duplicates by a stated key, and returns a new DataFrame. **Constraints:** do not mutate `raw` or use broad `dropna`; record conversion failures.
   **Verify:** assert raw preservation and `clean_frame(clean_frame(raw)).equals(clean_frame(raw))` for this contract.

3. Save only the cleaned frame under an ignored learner artifact directory, then read it back.
   **Expected behavior:** reloaded row count, columns, and key totals match the in-memory clean frame. **Constraints:** create parent folders with `Path.mkdir`, avoid absolute paths, and do not overwrite raw input.
   **Verify:** Read the saved file back and assert row count, ordered columns, dtypes/normalization policy, and selected key totals match the cleaned in-memory frame.

### Additional mastery practice

Profile before cleaning, preserve raw input, and make every conversion, imputation, and rejection rule observable and testable.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

4. **Prediction:** Predict the results of `pd.to_datetime(..., errors='coerce', utc=True)` for valid text, invalid text, and a timestamp with an offset.
   **Progressive hint:** Invalid text becomes `NaT`; valid values normalize to UTC.
   **Verify:** Create three input rows and assert valid/offset timestamps normalize to the expected UTC instants while invalid text becomes `NaT` and is counted.
5. **Tracing:** Trace conversion from object strings to pandas nullable `Int64`, including an empty value.
   **Progressive hint:** Nullable integer dtype can represent `<NA>` without becoming float.
   **Verify:** Record value and dtype before/after conversion; assert numeric strings become integers, empty input becomes `<NA>`, and dtype is nullable `Int64`.
6. **Implementation:** Implement a cleaner that normalizes column names, parses an event time, converts quantity, and returns a copy plus a quality summary.
   **Progressive hint:** Record invalid counts before dropping or imputing anything.
   **Verify:** Assert the cleaner leaves raw unchanged, returns expected normalized columns/dtypes, and reports exact invalid timestamp/quantity counts.
7. **Debugging:** Repair an in-place operation performed on a chained selection.
   **Progressive hint:** Use assignment on the owned copy and avoid `inplace=True` on a temporary object.
   **Verify:** Reproduce the warning/failure, then assert explicit owned-copy assignment changes only the intended frame and uses no `inplace` temporary mutation.
8. **Edge case and explanation:** Choose behavior for an all-missing numeric column whose median is also missing. Reject, use a domain default, or preserve missing, and justify.
   **Progressive hint:** A statistical fallback cannot be computed from zero observations.
   **Verify:** Run an all-missing fixture and assert the exact chosen reject/default/preserve policy; document why no sample median was available.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1: prediction, attempt, and evidence

**Contract reminder:** Load a local CSV into pandas, recording path, encoding, row grain, shape, columns, dtypes, missing counts, and duplicate-key counts before changing anything. **Expected behavior:** produce a compact profile, not a full data dump. **Constraint:** use repository-relative `Path` objects and no network source. **Verify:** save the first profile values, restart the kernel, and assert the second shape, columns, dtypes, missing counts, and duplicate-key counts exactly match.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 1: your work
# Task: Load a local CSV into pandas, recording path, encoding, row grain, shape, columns, dtypes, missing counts, and duplicate-key counts before changing anything. Constraint: use repository-relative `Path` objects and no network source.
# Verify: Produce a compact profile, not a full data dump. Save the first profile values, restart the kernel, and assert the second shape, columns, dtypes, missing counts, and duplicate-key counts exactly match.
# Prediction:

# Implementation:

# Verification:


### Practice 2: prediction, attempt, and evidence

**Contract reminder:** Implement `clean_frame(raw)` that trims selected text, converts documented numeric/date fields, handles missing values by written policy, resolves duplicates by a stated key, and returns a new DataFrame. **Constraints:** do not mutate `raw` or use broad `dropna`; record conversion failures. **Verify:** assert raw preservation and `clean_frame(clean_frame(raw)).equals(clean_frame(raw))` for this contract.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 2: your work
# Task: Implement `clean_frame(raw)` that trims selected text, converts documented numeric/date fields, handles missing values by written policy, resolves duplicates by a stated key, and returns a new DataFrame. Constraints: do not mutate `raw` or use broad `dropna`; record conversion failures.
# Verify: Assert raw preservation and `clean_frame(clean_frame(raw)).equals(clean_frame(raw))` for this contract.
# Prediction:

# Implementation:

# Verification:


### Practice 3: prediction, attempt, and evidence

**Contract reminder:** Save only the cleaned frame under an ignored learner artifact directory, then read it back. **Expected behavior:** reloaded row count, columns, and key totals match the in-memory clean frame. **Constraints:** create parent folders with `Path.mkdir`, avoid absolute paths, and do not overwrite raw input. **Verify:** Read the saved file back and assert row count, ordered columns, dtypes/normalization policy, and selected key totals match the cleaned in-memory frame.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 3: your work
# Task: Save only the cleaned frame under an ignored learner artifact directory, then read it back. Constraints: create parent folders with `Path.mkdir`, avoid absolute paths, and do not overwrite raw input.
# Verify: Reloaded row count, columns, and key totals match the in-memory clean frame. Read the saved file back and assert row count, ordered columns, dtypes/normalization policy, and selected key totals match the cleaned in-memory frame.
# Prediction:

# Implementation:

# Verification:


### Practice 4: prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict the results of `pd.to_datetime(..., errors='coerce', utc=True)` for valid text, invalid text, and a timestamp with an offset. **Progressive hint:** Invalid text becomes `NaT`; valid values normalize to UTC. **Verify:** Create three input rows and assert valid/offset timestamps normalize to the expected UTC instants while invalid text becomes `NaT` and is counted.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 4: your work
# Task: Predict the results of `pd.to_datetime(..., errors='coerce', utc=True)` for valid text, invalid text, and a timestamp with an offset.
# Verify: Create three input rows and assert valid/offset timestamps normalize to the expected UTC instants while invalid text becomes `NaT` and is counted.
# Prediction:

# Implementation:

# Verification:


### Practice 5: prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace conversion from object strings to pandas nullable `Int64`, including an empty value. **Progressive hint:** Nullable integer dtype can represent `<NA>` without becoming float. **Verify:** Record value and dtype before/after conversion; assert numeric strings become integers, empty input becomes `<NA>`, and dtype is nullable `Int64`.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 5: your work
# Task: Trace conversion from object strings to pandas nullable `Int64`, including an empty value.
# Verify: Record value and dtype before/after conversion; assert numeric strings become integers, empty input becomes `<NA>`, and dtype is nullable `Int64`.
# Prediction:

# Implementation:

# Verification:


### Practice 6: prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement a cleaner that normalizes column names, parses an event time, converts quantity, and returns a copy plus a quality summary. **Progressive hint:** Record invalid counts before dropping or imputing anything. **Verify:** Assert the cleaner leaves raw unchanged, returns expected normalized columns/dtypes, and reports exact invalid timestamp/quantity counts.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 6: your work
# Task: Implement a cleaner that normalizes column names, parses an event time, converts quantity, and returns a copy plus a quality summary.
# Verify: Assert the cleaner leaves raw unchanged, returns expected normalized columns/dtypes, and reports exact invalid timestamp/quantity counts.
# Prediction:

# Implementation:

# Verification:


### Practice 7: prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair an in-place operation performed on a chained selection. **Progressive hint:** Use assignment on the owned copy and avoid `inplace=True` on a temporary object. **Verify:** Reproduce the warning/failure, then assert explicit owned-copy assignment changes only the intended frame and uses no `inplace` temporary mutation.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 7: your work
# Task: Repair an in-place operation performed on a chained selection.
# Verify: Reproduce the warning/failure, then assert explicit owned-copy assignment changes only the intended frame and uses no `inplace` temporary mutation.
# Prediction:

# Implementation:

# Verification:


### Practice 8: prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Choose behavior for an all-missing numeric column whose median is also missing. Reject, use a domain default, or preserve missing, and justify. **Progressive hint:** A statistical fallback cannot be computed from zero observations. **Verify:** Run an all-missing fixture and assert the exact chosen reject/default/preserve policy; document why no sample median was available.

Write your prediction in the next cell before any code, then finish with the
check named in **Verify**. If it fails, keep the smallest failing input.

In [ ]:
# Practice 8: your work
# Task: Choose behavior for an all-missing numeric column whose median is also missing. Reject, use a domain default, or preserve missing, and justify.
# Verify: Run an all-missing fixture and assert the exact chosen reject/default/preserve policy; document why no sample median was available.
# Prediction:

# Implementation:

# Verification:
